# FairCredit: Algorithmic Bias Auditing and Mitigation in Digital Micro-Lending

**Purpose:** build a loan repayment/eligibility classifier, audit group fairness, test reweighing and threshold adjustment, and compare predictive performance with a simple illustrative business-value estimate.

> **Important dataset note:** `faircredit_synthetic_loan_dataset.csv` is synthetically generated for coursework. It is not real customer data and must not be presented as evidence about actual borrowers or as a production lending model. The target `repayment_success` is used as the positive outcome for this experiment. It is not historical loan approval.

## What this notebook produces
1. Baseline model
2. Reweighed model (pre-processing mitigation)
3. Group-specific threshold-adjusted predictions (post-processing experiment)
4. Proxy-variable association analysis
5. Fairness/accuracy comparison tables and plots
6. Exported CSV metrics and figures for the report


In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

RANDOM_STATE = 42
OUTPUT_DIR = "faircredit_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Imports ready.")


# Local fairness-metric helpers: avoids needing to install Fairlearn.
# These implement the subset of Fairlearn's MetricFrame interface used below.
def selection_rate(y_true, y_pred):
    y_pred = np.asarray(y_pred)
    return float(np.mean(y_pred == 1)) if len(y_pred) else np.nan

def true_positive_rate(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    positives = (y_true == 1)
    return float(np.mean(y_pred[positives] == 1)) if positives.any() else np.nan

def false_positive_rate(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    negatives = (y_true == 0)
    return float(np.mean(y_pred[negatives] == 1)) if negatives.any() else np.nan

class MetricFrame:
    def __init__(self, metrics, y_true, y_pred, sensitive_features):
        self.metrics = metrics
        self.y_true = np.asarray(y_true)
        self.y_pred = np.asarray(y_pred)
        self.sensitive_features = np.asarray(sensitive_features)
        rows = {}
        for group in pd.unique(self.sensitive_features):
            mask = self.sensitive_features == group
            rows[group] = {
                name: metric(self.y_true[mask], self.y_pred[mask])
                for name, metric in self.metrics.items()
            }
        self.by_group = pd.DataFrame.from_dict(rows, orient="index")


In [ ]:
# Load the dataset
# In Kaggle, add the CSV using Add Input, then update DATA_PATH to the mounted Kaggle path if needed.
DATA_PATH = "/kaggle/input/faircredit-dataset/faircredit_synthetic_loan_dataset.csv"
if not os.path.exists(DATA_PATH):
    DATA_PATH = "/kaggle/working/faircredit_synthetic_loan_dataset.csv"
if not os.path.exists(DATA_PATH):
    # In a fresh notebook, use the Kaggle upload panel or upload the provided CSV to /kaggle/working.
    raise FileNotFoundError(
        "Dataset not found. Upload faircredit_synthetic_loan_dataset.csv to Kaggle, "
        "or add it as a Kaggle Dataset and update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
display(df.describe(include="all").T)


In [ ]:
# Basic data quality checks
print("Missing values per column:")
display(df.isna().sum().to_frame("missing_count"))
print("Duplicate rows:", df.duplicated().sum())
print("Target balance:")
display(df["repayment_success"].value_counts(normalize=True).rename("proportion").to_frame())


In [ ]:
# Explore target and group representation
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="repayment_success", ax=axes[0])
axes[0].set_title("Repayment success target distribution")
sns.countplot(data=df, x="gender", hue="repayment_success", ax=axes[1])
axes[1].set_title("Target by gender")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/target_and_group_distribution.png", dpi=160, bbox_inches="tight")
plt.show()


## 1. Proxy-variable association analysis

We inspect association between the protected attributes (`gender`, `socioeconomic_group`) and potential proxies (`postal_code`, `occupation`). Cramér's V is used for categorical-to-categorical association. Association does **not** prove discrimination or causation; interpret alongside domain context.


In [ ]:
def cramers_v(x, y):
    tab = pd.crosstab(x, y)
    if tab.empty or min(tab.shape) < 2:
        return np.nan
    observed = tab.to_numpy()
    n_obs = observed.sum()
    expected = np.outer(observed.sum(axis=1), observed.sum(axis=0)) / n_obs
    chi2 = (((observed - expected) ** 2) / np.where(expected == 0, 1, expected)).sum()
    phi2 = chi2 / n_obs
    r, k = tab.shape
    # Bias-corrected Cramer's V
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n_obs-1))
    rcorr = r - ((r-1)**2)/(n_obs-1)
    kcorr = k - ((k-1)**2)/(n_obs-1)
    denom = min((kcorr-1), (rcorr-1))
    return np.sqrt(phi2corr / denom) if denom > 0 else np.nan

cat_cols = ["gender", "socioeconomic_group", "postal_code", "occupation"]
assoc = pd.DataFrame(index=cat_cols, columns=cat_cols, dtype=float)
for a in cat_cols:
    for b in cat_cols:
        assoc.loc[a, b] = 1.0 if a == b else cramers_v(df[a], df[b])
display(assoc.round(3))
plt.figure(figsize=(7, 5))
sns.heatmap(assoc.astype(float), annot=True, cmap="Blues", vmin=0, vmax=1)
plt.title("Categorical association matrix (Cramér's V)")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/proxy_association_heatmap.png", dpi=160, bbox_inches="tight")
plt.show()

# Numeric correlations are useful too, but should not be confused with categorical association.
numeric_cols = ["annual_income", "credit_score", "debt_to_income_ratio", "loan_amount", "employment_years", "previous_defaults"]
plt.figure(figsize=(9, 6))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Numeric feature correlation matrix")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/numeric_correlation_heatmap.png", dpi=160, bbox_inches="tight")
plt.show()


## 2. Define features, target and protected group

We predict **repayment success**, not historical loan approval. For the fairness audit, `gender` is the selected sensitive feature. `socioeconomic_group` is included in proxy analysis and can be used for a separate sensitivity audit. Sensitive attributes are excluded from model inputs here, but correlated proxies can still carry indirect information.


In [ ]:
TARGET = "repayment_success"
SENSITIVE = "gender"

# Do not use identifiers or the protected attribute as predictive features in this experiment.
DROP_COLS = ["applicant_id", TARGET, "gender"]
X = df.drop(columns=DROP_COLS)
y = df[TARGET].astype(int)
sensitive = df[SENSITIVE].astype(str)

X_train, X_test, y_train, y_test, s_train, s_test = train_test_split(
    X, y, sensitive, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features)
])

def make_model():
    return Pipeline([
        ("prep", preprocessor),
        ("model", LogisticRegression(max_iter=1500, class_weight=None, random_state=RANDOM_STATE))
    ])

print("Train rows:", len(X_train), "| Test rows:", len(X_test))
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


## 3. Train baseline model and calculate fairness metrics

For the positive outcome, `1` means repayment success. Approval-rate disparity is approximated here by selection rate for successful repayment predictions. In a real lender, the policy would need a separate, clearly defined approval target and validated repayment-risk model.


In [ ]:
baseline = make_model()
baseline.fit(X_train, y_train)
p_base = baseline.predict_proba(X_test)[:, 1]
pred_base = (p_base >= 0.50).astype(int)

def safe_auc(y_true, scores):
    return roc_auc_score(y_true, scores) if len(np.unique(y_true)) > 1 else np.nan

def metric_summary(name, y_true, y_pred, y_score, groups):
    mf = MetricFrame(
        metrics={
            "selection_rate": selection_rate,
            "TPR": true_positive_rate,
            "FPR": false_positive_rate,
            "accuracy": accuracy_score
        },
        y_true=y_true,
        y_pred=y_pred,
        sensitive_features=groups
    )
    by_group = mf.by_group
    tpr_gap = by_group["TPR"].max() - by_group["TPR"].min()
    fpr_gap = by_group["FPR"].max() - by_group["FPR"].min()
    sr = by_group["selection_rate"]
    # Ratio of smaller to larger selection rate; descriptive, not a legal conclusion.
    di = (sr.min() / sr.max()) if sr.max() > 0 else np.nan
    summary = {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": safe_auc(y_true, y_score),
        "selection_rate_gap": sr.max() - sr.min(),
        "disparate_impact_ratio": di,
        "TPR_gap": tpr_gap,
        "FPR_gap": fpr_gap,
        "equalized_odds_gap_max": max(tpr_gap, fpr_gap)
    }
    return summary, by_group

baseline_summary, baseline_groups = metric_summary("Baseline", y_test, pred_base, p_base, s_test)
display(pd.DataFrame([baseline_summary]).round(3))
display(baseline_groups.round(3))


## 4. Pre-processing mitigation: reweighing

Reweighing assigns weights to training examples based on the joint distribution of protected group and target outcome. We fit the same classifier using these sample weights. This is an educational implementation of the reweighing idea.


In [ ]:
train_frame = pd.DataFrame({"group": s_train.to_numpy(), "target": y_train.to_numpy()})
n_train = len(train_frame)
joint = train_frame.groupby(["group", "target"]).size() / n_train
group_prob = train_frame["group"].value_counts(normalize=True)
target_prob = train_frame["target"].value_counts(normalize=True)

weights = []
for g, yy in zip(s_train.to_numpy(), y_train.to_numpy()):
    p_joint = joint.loc[(g, yy)]
    weights.append((group_prob.loc[g] * target_prob.loc[yy]) / p_joint)
weights = np.asarray(weights, dtype=float)
weights = weights / weights.mean()

reweighed = make_model()
reweighed.fit(X_train, y_train, model__sample_weight=weights)
p_rew = reweighed.predict_proba(X_test)[:, 1]
pred_rew = (p_rew >= 0.50).astype(int)

reweighed_summary, reweighed_groups = metric_summary("Reweighed", y_test, pred_rew, p_rew, s_test)
display(pd.DataFrame([reweighed_summary]).round(3))
display(reweighed_groups.round(3))


## 5. Post-processing: group-specific threshold search

This experiment searches thresholds for each group using a validation subset from the training data, then evaluates the selected thresholds on the held-out test set. The search chooses thresholds that minimize the maximum difference in TPR and FPR, with a small penalty for moving far from 0.50. This is a demonstration, not a deployable lending policy. Group-specific decision thresholds can be legally restricted or inappropriate in real lending; consult applicable law and institutional policy.


In [ ]:
# Make a validation split from training data to avoid selecting thresholds on the test set.
X_fit, X_val, y_fit, y_val, s_fit, s_val = train_test_split(
    X_train, y_train, s_train, test_size=0.25, random_state=RANDOM_STATE, stratify=y_train
)
val_model = make_model()
val_model.fit(X_fit, y_fit)
p_val = val_model.predict_proba(X_val)[:, 1]

groups_unique = sorted(s_val.unique())
threshold_grid = np.arange(0.20, 0.81, 0.02)

def rates(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    tpr = tp / (tp + fn) if (tp + fn) else 0.0
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    return tpr, fpr

best_pair = None
best_obj = np.inf
for t0 in threshold_grid:
    for t1 in threshold_grid:
        thresholds = {groups_unique[0]: t0, groups_unique[1]: t1}
        val_pred = np.array([int(score >= thresholds[g]) for score, g in zip(p_val, s_val)])
        group_rates = {}
        for g in groups_unique:
            mask = (s_val.to_numpy() == g)
            group_rates[g] = rates(y_val.to_numpy()[mask], val_pred[mask])
        tpr_gap = abs(group_rates[groups_unique[0]][0] - group_rates[groups_unique[1]][0])
        fpr_gap = abs(group_rates[groups_unique[0]][1] - group_rates[groups_unique[1]][1])
        # Tiny regularization favors thresholds near 0.50 when fairness is similar.
        obj = max(tpr_gap, fpr_gap) + 0.02 * (abs(t0-0.5) + abs(t1-0.5))
        if obj < best_obj:
            best_obj, best_pair = obj, thresholds

print("Selected validation thresholds:", best_pair)
p_cal = baseline.predict_proba(X_test)[:, 1]
pred_cal = np.array([int(score >= best_pair[g]) for score, g in zip(p_cal, s_test)])

cal_summary, cal_groups = metric_summary("Threshold-adjusted", y_test, pred_cal, p_cal, s_test)
display(pd.DataFrame([cal_summary]).round(3))
display(cal_groups.round(3))


## 6. Compare all models

Metrics are calculated on the same held-out test rows. A lower fairness gap is generally preferable for the selected metric, but no single metric captures every aspect of fairness. Review accuracy, precision, recall and group-specific error rates together.


In [ ]:
summaries = pd.DataFrame([baseline_summary, reweighed_summary, cal_summary]).set_index("model")
display(summaries.round(3))
summaries.to_csv(f"{OUTPUT_DIR}/model_comparison_metrics.csv")

plot_cols = ["accuracy", "precision", "recall", "f1", "disparate_impact_ratio"]
summaries[plot_cols].plot(kind="bar", figsize=(12, 5), ylim=(0, 1.05))
plt.title("Predictive performance and selection-rate ratio")
plt.ylabel("Score / ratio")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/performance_and_fairness_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

gap_cols = ["selection_rate_gap", "TPR_gap", "FPR_gap", "equalized_odds_gap_max"]
summaries[gap_cols].plot(kind="bar", figsize=(12, 5))
plt.title("Fairness gap comparison (lower is smaller disparity)")
plt.ylabel("Absolute gap")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fairness_gap_comparison.png", dpi=160, bbox_inches="tight")
plt.show()


## 7. Group-level results and confusion matrices

In [ ]:
group_tables = {}
for name, pred in [("Baseline", pred_base), ("Reweighed", pred_rew), ("Threshold-adjusted", pred_cal)]:
    mf = MetricFrame(
        metrics={"selection_rate": selection_rate, "TPR": true_positive_rate, "FPR": false_positive_rate, "accuracy": accuracy_score},
        y_true=y_test, y_pred=pred, sensitive_features=s_test
    )
    group_tables[name] = mf.by_group
    print("\\n", name)
    display(mf.by_group.round(3))
    print("Confusion matrix:")
    display(pd.DataFrame(confusion_matrix(y_test, pred, labels=[0,1]),
                         index=["Actual 0", "Actual 1"], columns=["Predicted 0", "Predicted 1"]))


## 8. Illustrative business viability analysis

This is a deliberately simple scenario analysis, not a validated financial forecast. We treat a positive prediction as an applicant predicted to repay successfully. The assumptions below are placeholders; replace them with instructor-approved values. A real analysis would account for loan amount, interest revenue, default loss, funding cost, operating costs, regulations and customer impact.


In [ ]:
# Illustrative assumptions in arbitrary currency units; replace for your report.
REVENUE_PER_SUCCESSFUL_LOAN = 1200
LOSS_PER_UNSUCCESSFUL_LOAN = 5000
OPERATING_COST_PER_APPROVED_LOAN = 250

def business_estimate(name, predictions, y_true):
    approved = predictions == 1
    n_approved = int(approved.sum())
    if n_approved == 0:
        success_rate = np.nan
        expected_value = 0.0
    else:
        success_rate = float(np.mean(y_true.to_numpy()[approved]))
        # Approximate value using realized target outcomes in test data, only as a retrospective toy calculation.
        values = np.where(y_true.to_numpy()[approved] == 1,
                          REVENUE_PER_SUCCESSFUL_LOAN,
                          -LOSS_PER_UNSUCCESSFUL_LOAN) - OPERATING_COST_PER_APPROVED_LOAN
        expected_value = float(values.sum())
    return {"model": name, "predicted_positive_count": n_approved,
            "predicted_positive_rate": float(approved.mean()),
            "observed_success_rate_among_positive": success_rate,
            "illustrative_net_value": expected_value}

business = pd.DataFrame([
    business_estimate("Baseline", pred_base, y_test),
    business_estimate("Reweighed", pred_rew, y_test),
    business_estimate("Threshold-adjusted", pred_cal, y_test)
]).set_index("model")
display(business.round(3))
business.to_csv(f"{OUTPUT_DIR}/illustrative_business_comparison.csv")
print("Important: these figures are illustrative and based on synthetic test outcomes; do not describe them as real bank profit.")


## 9. Export results for your report

The notebook saves tables and charts to `faircredit_outputs/`. Download these from the Kaggle notebook's Output/Files panel after running all cells.


In [ ]:
# Export group tables and model predictions
predictions_export = pd.DataFrame({
    "y_true_repayment_success": y_test.to_numpy(),
    "gender_group": s_test.to_numpy(),
    "baseline_probability": p_base,
    "baseline_prediction": pred_base,
    "reweighed_probability": p_rew,
    "reweighed_prediction": pred_rew,
    "threshold_adjusted_probability": p_cal,
    "threshold_adjusted_prediction": pred_cal
})
predictions_export.to_csv(f"{OUTPUT_DIR}/test_predictions.csv", index=False)
assoc.to_csv(f"{OUTPUT_DIR}/proxy_association_matrix.csv")
print("Files created:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    print("-", f)


## 10. Report discussion prompts

Use your actual outputs to answer these questions:
1. Which proxy features were most associated with gender or socioeconomic group? Why does association not prove discrimination?
2. How did the baseline selection-rate ratio and Equalized Odds gap compare across groups?
3. Did reweighing improve fairness? What happened to accuracy, precision and recall?
4. Did the threshold-adjusted model reduce TPR/FPR gaps on the test set? Was the change meaningful?
5. Which fairness metric improved and which did not? Why might fairness metrics conflict?
6. How did the number of predicted-positive applicants and illustrative value change?
7. What data, legal, privacy and monitoring limitations would need to be addressed before real deployment?

**Limitations to disclose:** synthetic data; one selected sensitive attribute; simplified profit assumptions; possible sampling variability; proxy correlation is not causal proof; group-specific thresholds may be legally or ethically inappropriate in real lending. This is a coursework demonstration, not a real credit decision system.
